# Run experiments on BKT with Junyi dataset

## Imports

In [ ]:
!pip -q install git+https://github.com/CAHLR/pyBKT.git pandas numpy scikit-learn

from google.colab import drive
drive.mount("/content/drive")

import os
import time
import random
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, brier_score_loss
from pyBKT.models import Model

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BASE_DIR = "/content/drive/MyDrive/education-ml-research"

print("Setup complete.")
print("Project directory exists:", os.path.exists(BASE_DIR))

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Mounted at /content/drive
Setup complete.
Project directory exists: True


## Load dataset and IRT groups

In [ ]:
DATA_PATH_JUNYI = os.path.join(BASE_DIR, "Junyi", "Log_Problem.csv")
GROUPS_PATH_JUNYI = os.path.join(BASE_DIR, "Junyi", "student_irt_ability_groups.csv")

junyi_raw = pd.read_csv(DATA_PATH_JUNYI)
irt_groups_junyi = pd.read_csv(GROUPS_PATH_JUNYI)

print("Dataset shape:", junyi_raw.shape)
print("IRT groups shape:", irt_groups_junyi.shape)

print("\nDataset columns:")
print(junyi_raw.columns.tolist())

print("\nDataset preview:")
display(junyi_raw.head())

print("\nIRT groups columns:")
print(irt_groups_junyi.columns.tolist())

print("\nIRT groups preview:")
display(irt_groups_junyi.head())

Dataset shape: (16217311, 14)
IRT groups shape: (72758, 4)

Dataset columns:
['timestamp_TW', 'uuid', 'ucid', 'upid', 'problem_number', 'exercise_problem_repeat_session', 'is_correct', 'total_sec_taken', 'total_attempt_cnt', 'used_hint_cnt', 'is_hint_used', 'is_downgrade', 'is_upgrade', 'level']

Dataset preview:


,timestamp_TW,uuid,ucid,upid,problem_number,exercise_problem_repeat_session,is_correct,total_sec_taken,total_attempt_cnt,used_hint_cnt,is_hint_used,is_downgrade,is_upgrade,level
0,2019-05-26 21:00:00 UTC,FLy+lviglNR5Y1l0Xiijnl6QHySBcpKHJLCtQ6ogm2Q=,KDOmuTrY/IJzDP4kIgIYCBiGyTymsJ8Iy4cDB35WGYg=,Vbs92l4JmdiWkUEm/iahxnUTaac2oN1IlUtXB7JcfoE=,18,2,True,33,1,0,False,False,True,3
1,2019-05-17 16:30:00 UTC,+Gqj2nalc6M9fusyVECTC0AN7UQdDQTXESIuElkDltU=,COZ39Wo+uIUO2s7c2VGEHjJf6Vx0xifxVAiaeHtaTdk=,Ek+pIeHNNoEo0tGEq91eBcBmGgy3+A5RWhpj95zTyHM=,4,1,True,8,1,0,False,NaN,NaN,0
2,2019-05-15 19:15:00 UTC,6D5QN8j8ng/VR74ES3A0zqAj0bIFFyaKjKEj8ZyXjQ8=,TwyqyV1uJYlDAX8wX/PtTCVZEBo/APIVfTzzleGkNCQ=,1MBa2f5Qog4JBoAuUfJf0fxeJctdEirAqKgfsg246eI=,9,1,True,17,1,0,False,NaN,NaN,0
3,2019-05-05 14:45:00 UTC,GgTZuCqZXObthtK6GAwqvlHrTMm5pKHWeezQxL/pcKc=,tBo6ECyT8IlKAM8UhQHWkqv92PRLcSiwuerfC7vNX+w=,kdMy2nG+QVMjPkuaMEWs0yV/sYZVoG1vm7zM0fCy+qk=,2,1,True,10,1,0,False,NaN,NaN,0
4,2019-05-14 16:45:00 UTC,JMNKWoU0CkMSzgQ8bCnmCYlD8jEzAVge3lHMYLXKM2g=,vVpSKAMQbTMvtdERR0ksOeRmmaFt0R210t4Z//0RpPA=,jjPR8fmkLSFoCQQYB4g6kI8mgdcK3sKtMirKUvfmZIk=,6,1,True,98,1,0,False,NaN,NaN,0



IRT groups columns:
['uuid', 'theta_all', 'theta_quartile', 'theta_sd_group']

IRT groups preview:


,uuid,theta_all,theta_quartile,theta_sd_group
0,++5bdNp/LZvGenJ8Brp4n2SfS9d4pu4qA7cF7FQW7hk=,3.09589,Q4,> +1.5 SD
1,++9EkR6syMGk44XpyhOj40cg9xiXwCLS/TwEy+ujrL0=,-2.32968,Q1,< -1.5 SD
2,++E4TrlDYvGtPBg1edhkLXLEEbnfiAgAamPQ33vpW8M=,-0.63297,Q2,-1.0 to -0.5 SD
3,++G4mkLfs4WDYhc1Ga+3G+/oqSniQQvLBm7SBQ3V39Y=,-0.26564,Q2,-0.5 to 0 SD
4,++GobOSWqrsaxoRg1bMN+T6biIJcgBXwuOH/ddq3DiU=,0.57712,Q3,+0.5 to +1.0 SD


## Check data quality and key columns

In [ ]:
print("=== Dataset checks ===")
print("Unique students:", junyi_raw["uuid"].nunique())
print("Unique skills/concepts (ucid):", junyi_raw["ucid"].nunique())
print("Unique problems (upid):", junyi_raw["upid"].nunique())

print("\nCorrectness distribution:")
print(junyi_raw["is_correct"].value_counts(dropna=False))
print("\nCorrectness proportions:")
print(junyi_raw["is_correct"].value_counts(normalize=True, dropna=False).round(4))

print("\nMissing values in key columns:")
print(
    junyi_raw[["uuid", "ucid", "is_correct", "timestamp_TW"]]
    .isna()
    .sum()
)

print("\n=== IRT group checks ===")
print("Unique IRT students:", irt_groups_junyi["uuid"].nunique())
print("Missing IRT ability estimates:", irt_groups_junyi["theta_all"].isna().sum())

print("\nIRT quartile counts:")
print(irt_groups_junyi["theta_quartile"].value_counts().sort_index())

print("\nIRT SD-group counts:")
print(irt_groups_junyi["theta_sd_group"].value_counts().sort_index())

print("\n=== Student overlap ===")
dataset_students = set(junyi_raw["uuid"].dropna().unique())
irt_students = set(irt_groups_junyi["uuid"].dropna().unique())

print("Students in both files:", len(dataset_students & irt_students))
print("Dataset students without IRT estimates:", len(dataset_students - irt_students))
print("IRT students absent from dataset:", len(irt_students - dataset_students))

=== Dataset checks ===
Unique students: 72758
Unique skills/concepts (ucid): 1326
Unique problems (upid): 25785

Correctness distribution:
is_correct
True     11412558
False     4804753
Name: count, dtype: int64

Correctness proportions:
is_correct
True    0.70370
False   0.29630
Name: proportion, dtype: float64

Missing values in key columns:
uuid            0
ucid            0
is_correct      0
timestamp_TW    0
dtype: int64

=== IRT group checks ===
Unique IRT students: 72758
Missing IRT ability estimates: 0

IRT quartile counts:
theta_quartile
Q1    18190
Q2    18189
Q3    18189
Q4    18190
Name: count, dtype: int64

IRT SD-group counts:
theta_sd_group
+0.5 to +1.0 SD     9692
+1.0 to +1.5 SD     5169
-0.5 to 0 SD       15274
-1.0 to -0.5 SD    11968
-1.5 to -1.0 SD     7060
0 to +0.5 SD       14323
< -1.5 SD           5423
> +1.5 SD           3849
Name: count, dtype: int64

=== Student overlap ===
Students in both files: 72758
Dataset students without IRT estimates: 0
IRT students

## Prepare interaction data for BKT

In [ ]:
junyi = junyi_raw[
    ["uuid", "ucid", "is_correct", "timestamp_TW"]
].copy()

junyi = junyi.rename(columns={
    "uuid": "user_id",
    "ucid": "skill_id",
    "is_correct": "correct",
    "timestamp_TW": "timestamp"
})

junyi["correct"] = junyi["correct"].astype("int8")
junyi["skill_id"] = junyi["skill_id"].astype(str)

# Sort chronologically within each student.
# Original row order breaks ties when timestamps are identical.
junyi["_original_order"] = np.arange(len(junyi), dtype=np.int64)
junyi = junyi.sort_values(
    ["user_id", "timestamp", "_original_order"]
).reset_index(drop=True)

# Assign sequential interaction indices within each student.
junyi["order_id"] = (
    junyi.groupby("user_id", sort=False).cumcount().astype("int32")
)

junyi = junyi[
    ["order_id", "user_id", "skill_id", "correct"]
]

print("Prepared interactions:", len(junyi))
print("Unique students:", junyi["user_id"].nunique())
print("Unique skills:", junyi["skill_id"].nunique())
print("Correct response rate:", round(junyi["correct"].mean(), 5))
print("\nPrepared data preview:")
display(junyi.head())

Prepared interactions: 16217311
Unique students: 72758
Unique skills: 1326
Correct response rate: 0.70373

Prepared data preview:


,order_id,user_id,skill_id,correct
0,0,++5bdNp/LZvGenJ8Brp4n2SfS9d4pu4qA7cF7FQW7hk=,VY6aXT7f64ny+uy4pszHVNSy3WHyoFPuhwToxBhB3wM=,1
1,1,++5bdNp/LZvGenJ8Brp4n2SfS9d4pu4qA7cF7FQW7hk=,VY6aXT7f64ny+uy4pszHVNSy3WHyoFPuhwToxBhB3wM=,1
2,2,++5bdNp/LZvGenJ8Brp4n2SfS9d4pu4qA7cF7FQW7hk=,VY6aXT7f64ny+uy4pszHVNSy3WHyoFPuhwToxBhB3wM=,1
3,3,++5bdNp/LZvGenJ8Brp4n2SfS9d4pu4qA7cF7FQW7hk=,VY6aXT7f64ny+uy4pszHVNSy3WHyoFPuhwToxBhB3wM=,1
4,4,++5bdNp/LZvGenJ8Brp4n2SfS9d4pu4qA7cF7FQW7hk=,VY6aXT7f64ny+uy4pszHVNSy3WHyoFPuhwToxBhB3wM=,1


## Create train/test split

In [ ]:
SEED_JUNYI = 43

# Map each student to their existing IRT quartile
student_groups_junyi = irt_groups_junyi[
    ["uuid", "theta_all", "theta_quartile", "theta_sd_group"]
].copy()

train_students_junyi = []
test_students_junyi = []

for quartile in ["Q1", "Q2", "Q3", "Q4"]:
    students = student_groups_junyi.loc[
        student_groups_junyi["theta_quartile"] == quartile,
        "uuid"
    ].to_numpy()

    train_ids, test_ids = train_test_split(
        students,
        test_size=0.20,
        random_state=SEED_JUNYI
    )

    train_students_junyi.extend(train_ids)
    test_students_junyi.extend(test_ids)

train_ids_junyi = set(train_students_junyi)
test_ids_junyi = set(test_students_junyi)

print("Training students:", len(train_ids_junyi))
print("Testing students:", len(test_ids_junyi))
print("Student overlap:", len(train_ids_junyi & test_ids_junyi))

print("\nTraining students by quartile:")
print(
    student_groups_junyi[
        student_groups_junyi["uuid"].isin(train_ids_junyi)
    ]["theta_quartile"].value_counts().sort_index()
)

print("\nTesting students by quartile:")
print(
    student_groups_junyi[
        student_groups_junyi["uuid"].isin(test_ids_junyi)
    ]["theta_quartile"].value_counts().sort_index()
)

Training students: 58206
Testing students: 14552
Student overlap: 0

Training students by quartile:
theta_quartile
Q1    14552
Q2    14551
Q3    14551
Q4    14552
Name: count, dtype: int64

Testing students by quartile:
theta_quartile
Q1    3638
Q2    3638
Q3    3638
Q4    3638
Name: count, dtype: int64


## Verify the split

In [ ]:
train_mask_junyi = junyi["user_id"].isin(train_ids_junyi)

junyi_train = junyi.loc[train_mask_junyi].copy()
junyi_test = junyi.loc[~train_mask_junyi].copy()

train_skills_junyi = set(junyi_train["skill_id"].unique())
test_skills_junyi = set(junyi_test["skill_id"].unique())
unseen_skills_junyi = test_skills_junyi - train_skills_junyi

print("Training interactions:", len(junyi_train))
print("Testing interactions:", len(junyi_test))
print("Training students:", junyi_train["user_id"].nunique())
print("Testing students:", junyi_test["user_id"].nunique())
print("Training skills:", len(train_skills_junyi))
print("Testing skills:", len(test_skills_junyi))
print("Test skills unseen during training:", len(unseen_skills_junyi))

print("\nTraining correctness rate:", round(junyi_train["correct"].mean(), 5))
print("Testing correctness rate:", round(junyi_test["correct"].mean(), 5))

Training interactions: 12924917
Testing interactions: 3292394
Training students: 58206
Testing students: 14552
Training skills: 1326
Testing skills: 1326
Test skills unseen during training: 0

Training correctness rate: 0.70376
Testing correctness rate: 0.70359


## Fit the model

In [ ]:
bkt_train_junyi = junyi_train[
    ["order_id", "user_id", "skill_id", "correct"]
].copy()

bkt_test_junyi = junyi_test[
    ["order_id", "user_id", "skill_id", "correct"]
].copy()

start = time.time()

bkt_model_junyi = Model(
    seed=42,
    num_fits=1,
    parallel=False
)

bkt_model_junyi.fit(
    data=bkt_train_junyi,
    defaults={
        "order_id": "order_id",
        "student_id": "user_id",
        "skill_name": "skill_id",
        "correct": "correct"
    }
)

print(f"BKT training time: {(time.time() - start) / 60:.2f} minutes")
print("Skills fitted:", len(train_skills_junyi))
print("BKT training complete.")

BKT training time: 66.02 minutes
Skills fitted: 1326
BKT training complete.


In [ ]:
print("Skills fitted:", len(bkt_model_junyi.fit_model))
print("Training students:", junyi_train["user_id"].nunique())
print("Test students:", junyi_test["user_id"].nunique())
print("Test interactions:", len(junyi_test))
print("Train/test student overlap:",
      len(train_ids_junyi & test_ids_junyi))

Skills fitted: 1326
Training students: 58206
Test students: 14552
Test interactions: 3292394
Train/test student overlap: 0


## Predict on test data

In [ ]:
# Create safe identifiers for the skills already fitted by BKT.
old_skills = list(bkt_model_junyi.fit_model.keys())

skill_mapping_junyi = {
    old_skill: f"skill_{i:04d}"
    for i, old_skill in enumerate(old_skills)
}

# Rename the fitted-model dictionary keys without changing fitted parameters.
bkt_model_junyi.fit_model = {
    skill_mapping_junyi[skill]: params
    for skill, params in bkt_model_junyi.fit_model.items()
}

# Update the model's skill list and test-data skill IDs consistently.
bkt_model_junyi.skills = [
    skill_mapping_junyi[skill]
    for skill in old_skills
]

bkt_test_junyi = bkt_test_junyi.copy()
bkt_test_junyi["skill_id"] = (
    bkt_test_junyi["skill_id"].map(skill_mapping_junyi)
)

print("Fitted skills:", len(bkt_model_junyi.fit_model))
print("Unmapped test interactions:", bkt_test_junyi["skill_id"].isna().sum())
print("Unique test skills:", bkt_test_junyi["skill_id"].nunique())
print("Example mapped skills:", bkt_test_junyi["skill_id"].unique()[:5])

Fitted skills: 1326
Unmapped test interactions: 0
Unique test skills: 1326
Example mapped skills: ['skill_0117' 'skill_0917' 'skill_0639' 'skill_0885' 'skill_0752']


In [ ]:
start = time.time()

bkt_predictions_junyi = bkt_model_junyi.predict(
    data=bkt_test_junyi
)

print(f"Prediction time: {(time.time() - start) / 60:.2f} minutes")
print("Prediction rows:", len(bkt_predictions_junyi))
print("Prediction columns:", bkt_predictions_junyi.columns.tolist())
print(bkt_predictions_junyi.head())

Prediction time: 7.90 minutes
Prediction rows: 3292394
Prediction columns: ['order_id', 'user_id', 'skill_id', 'correct', 'correct_predictions', 'state_predictions']
    order_id                                       user_id    skill_id  \
44         0  ++G4mkLfs4WDYhc1Ga+3G+/oqSniQQvLBm7SBQ3V39Y=  skill_0117   
45         1  ++G4mkLfs4WDYhc1Ga+3G+/oqSniQQvLBm7SBQ3V39Y=  skill_0917   
46         2  ++G4mkLfs4WDYhc1Ga+3G+/oqSniQQvLBm7SBQ3V39Y=  skill_0917   
47         3  ++G4mkLfs4WDYhc1Ga+3G+/oqSniQQvLBm7SBQ3V39Y=  skill_0917   
48         4  ++G4mkLfs4WDYhc1Ga+3G+/oqSniQQvLBm7SBQ3V39Y=  skill_0917   

    correct  correct_predictions  state_predictions  
44        0              0.35698            0.47673  
45        0              0.45290            0.50001  
46        0              0.37817            0.32947  
47        0              0.32530            0.20882  
48        0              0.29498            0.13962  


## Predict on train data

In [ ]:
# Apply the same skill mapping used for the test data.
bkt_train_junyi = bkt_train_junyi.copy()

bkt_train_junyi["skill_id"] = (
    bkt_train_junyi["skill_id"].map(skill_mapping_junyi)
)

print("Training interactions:", len(bkt_train_junyi))
print("Unmapped training interactions:", bkt_train_junyi["skill_id"].isna().sum())
print("Unique training skills:", bkt_train_junyi["skill_id"].nunique())
print("Missing values:", bkt_train_junyi["skill_id"].isna().sum())

Training interactions: 12924917
Unmapped training interactions: 0
Unique training skills: 1326
Missing values: 0


In [ ]:
import time

start = time.time()

bkt_train_predictions_junyi = bkt_model_junyi.predict(
    data=bkt_train_junyi
)

print(f"Training prediction time: {(time.time() - start) / 60:.2f} minutes")
print("Training prediction rows:", len(bkt_train_predictions_junyi))
print("Prediction columns:", bkt_train_predictions_junyi.columns.tolist())
print(bkt_train_predictions_junyi.head())

Training prediction time: 30.90 minutes
Training prediction rows: 12924917
Prediction columns: ['order_id', 'user_id', 'skill_id', 'correct', 'correct_predictions', 'state_predictions']
   order_id                                       user_id    skill_id  \
0         0  ++5bdNp/LZvGenJ8Brp4n2SfS9d4pu4qA7cF7FQW7hk=  skill_0000   
1         1  ++5bdNp/LZvGenJ8Brp4n2SfS9d4pu4qA7cF7FQW7hk=  skill_0000   
2         2  ++5bdNp/LZvGenJ8Brp4n2SfS9d4pu4qA7cF7FQW7hk=  skill_0000   
3         3  ++5bdNp/LZvGenJ8Brp4n2SfS9d4pu4qA7cF7FQW7hk=  skill_0000   
4         4  ++5bdNp/LZvGenJ8Brp4n2SfS9d4pu4qA7cF7FQW7hk=  skill_0000   

   correct  correct_predictions  state_predictions  
0        1              0.59332            0.53017  
1        1              0.66018            0.69821  
2        1              0.71094            0.82578  
3        1              0.74309            0.90661  
4        1              0.76120            0.95210  


## Calculate AUC and Brier Scores

In [ ]:
from sklearn.metrics import roc_auc_score, brier_score_loss

y_train_true_junyi = bkt_train_predictions_junyi["correct"].astype(int)
y_train_pred_junyi = (
    bkt_train_predictions_junyi["correct_predictions"].astype(float)
)

junyi_bkt_train_auc = roc_auc_score(
    y_train_true_junyi, y_train_pred_junyi
)
junyi_bkt_train_brier = brier_score_loss(
    y_train_true_junyi, y_train_pred_junyi
)

print("Junyi BKT — Overall Training Results (In-Sample)")
print(f"Training interactions: {len(bkt_train_predictions_junyi):,}")
print(f"Observed correct rate: {y_train_true_junyi.mean():.6f}")
print(f"Mean predicted probability: {y_train_pred_junyi.mean():.6f}")
print(f"Training AUC: {junyi_bkt_train_auc:.6f}")
print(f"Training Brier Score: {junyi_bkt_train_brier:.6f}")

Junyi BKT — Overall Training Results (In-Sample)
Training interactions: 12,924,917
Observed correct rate: 0.703763
Mean predicted probability: 0.704713
Training AUC: 0.732624
Training Brier Score: 0.177809


In [ ]:
import pandas as pd
from sklearn.metrics import roc_auc_score, brier_score_loss

# Attach IRT ability groups to test predictions.
junyi_test_eval = bkt_predictions_junyi.merge(
    student_groups_junyi[["uuid", "theta_quartile", "theta_sd_group"]],
    left_on="user_id",
    right_on="uuid",
    how="left",
    validate="many_to_one"
)

if len(junyi_test_eval) != len(bkt_predictions_junyi):
    raise ValueError("Row count changed during merge.")

if junyi_test_eval["theta_quartile"].isna().any():
    raise ValueError("Some test predictions are missing IRT group assignments.")

# Calculate metrics for each group.
def calculate_group_metrics(df, group_col):
    results = []

    for group, subset in df.groupby(group_col, observed=True, sort=True):
        y_true = subset["correct"].astype(int)
        y_pred = subset["correct_predictions"].astype(float)

        results.append({
            "Group": group,
            "Students": subset["user_id"].nunique(),
            "Interactions": len(subset),
            "Correct Rate": y_true.mean(),
            "Mean Predicted Probability": y_pred.mean(),
            "AUC": (
                roc_auc_score(y_true, y_pred)
                if y_true.nunique() == 2 else float("nan")
            ),
            "Brier Score": brier_score_loss(y_true, y_pred)
        })

    return pd.DataFrame(results)

quartile_results_junyi = calculate_group_metrics(
    junyi_test_eval, "theta_quartile"
)

sd_results_junyi = calculate_group_metrics(
    junyi_test_eval, "theta_sd_group"
)

print("JUNYI BKT — HELD-OUT TEST RESULTS BY IRT QUARTILE")
display(quartile_results_junyi.round(6))

print("\nJUNYI BKT — HELD-OUT TEST RESULTS BY IRT STANDARD-DEVIATION GROUP")
display(sd_results_junyi.round(6))

print("\nValidation")
print("Test prediction rows:", len(bkt_predictions_junyi))
print("Evaluation rows:", len(junyi_test_eval))
print("Unique test students:", junyi_test_eval["user_id"].nunique())
print("Missing quartile assignments:",
      junyi_test_eval["theta_quartile"].isna().sum())
print("Missing SD-group assignments:",
      junyi_test_eval["theta_sd_group"].isna().sum())

JUNYI BKT — HELD-OUT TEST RESULTS BY IRT QUARTILE


,Group,Students,Interactions,Correct Rate,Mean Predicted Probability,AUC,Brier Score
0,Q1,3638,727044,0.55335,0.65362,0.72419,0.22043
1,Q2,3638,1044715,0.68219,0.70248,0.72709,0.18709
2,Q3,3638,929252,0.76317,0.72837,0.72352,0.16050
3,Q4,3638,591383,0.83244,0.73660,0.70544,0.13838



JUNYI BKT — HELD-OUT TEST RESULTS BY IRT STANDARD-DEVIATION GROUP


,Group,Students,Interactions,Correct Rate,Mean Predicted Probability,AUC,Brier Score
0,+0.5 to +1.0 SD,1984,436012,0.81415,0.73587,0.70992,0.14420
1,+1.0 to +1.5 SD,1000,129706,0.85032,0.73568,0.69852,0.13375
2,-0.5 to 0 SD,3094,881300,0.70305,0.70959,0.72701,0.18061
3,-1.0 to -0.5 SD,2327,633223,0.63437,0.68521,0.72512,0.20133
4,-1.5 to -1.0 SD,1434,295923,0.54331,0.64775,0.72453,0.22200
5,0 to +0.5 SD,2860,732680,0.76601,0.72824,0.72229,0.15984
6,< -1.5 SD,1082,124925,0.43591,0.61681,0.71540,0.24482
7,> +1.5 SD,771,58625,0.91147,0.74581,0.68816,0.10949



Validation
Test prediction rows: 3292394
Evaluation rows: 3292394
Unique test students: 14552
Missing quartile assignments: 0
Missing SD-group assignments: 0


## Junyi BKT Results

### Overall Training Results (In-Sample)

| Dataset | Model | Split | Interactions | AUC | Brier Score |
|---|---|---|---:|---:|---:|
| Junyi | BKT | Train (in-sample) | 12,924,917 | 0.732624 | 0.177809 |

### Held-Out Test Results by IRT Quartile

| Dataset | Model | Ability Group | Students | Interactions | Correct Rate | Mean Predicted Probability | AUC | Brier Score |
|---|---|---|---:|---:|---:|---:|---:|---:|
| Junyi | BKT | Q1 | 3,638 | 727,044 | 0.55335 | 0.65362 | 0.72419 | 0.22043 |
| Junyi | BKT | Q2 | 3,638 | 1,044,715 | 0.68219 | 0.70248 | 0.72709 | 0.18709 |
| Junyi | BKT | Q3 | 3,638 | 929,252 | 0.76317 | 0.72837 | 0.72352 | 0.16050 |
| Junyi | BKT | Q4 | 3,638 | 591,383 | 0.83244 | 0.73660 | 0.70544 | 0.13838 |

### Held-Out Test Results by IRT Standard-Deviation Group

| Dataset | Model | Ability Group | Students | Interactions | Correct Rate | Mean Predicted Probability | AUC | Brier Score |
|---|---|---|---:|---:|---:|---:|---:|---:|
| Junyi | BKT | < -1.5 SD | 1,082 | 124,925 | 0.43591 | 0.61681 | 0.71540 | 0.24482 |
| Junyi | BKT | -1.5 to -1.0 SD | 1,434 | 295,923 | 0.54331 | 0.64775 | 0.72453 | 0.22200 |
| Junyi | BKT | -1.0 to -0.5 SD | 2,327 | 633,223 | 0.63437 | 0.68521 | 0.72512 | 0.20133 |
| Junyi | BKT | -0.5 to 0 SD | 3,094 | 881,300 | 0.70305 | 0.70959 | 0.72701 | 0.18061 |
| Junyi | BKT | 0 to +0.5 SD | 2,860 | 732,680 | 0.76601 | 0.72824 | 0.72229 | 0.15984 |
| Junyi | BKT | +0.5 to +1.0 SD | 1,984 | 436,012 | 0.81415 | 0.73587 | 0.70992 | 0.14420 |
| Junyi | BKT | +1.0 to +1.5 SD | 1,000 | 129,706 | 0.85032 | 0.73568 | 0.69852 | 0.13375 |
| Junyi | BKT | > +1.5 SD | 771 | 58,625 | 0.91147 | 0.74581 | 0.68816 | 0.10949 |